# The CF0360 cubed-sphere grid, catchments and tiles

This notebook shows how the three spatial layers of the CF0360 GEOSldas configuration fit together, and how
observations end up on tiles.

1. **The cubed-sphere grid (`CF0360x6C`).** The globe is covered by the 6 faces of a cube projected onto the
   sphere, and each face is split into 360 × 360 cells. That gives 6 × 360 × 360 = 777,600 cells of about
   28 km (roughly 0.25°). Unlike a lat/lon grid, cells stay about the same size everywhere and there are no
   poles where the cells squeeze together. The price is that grid lines are not lat/lon lines: they bend,
   and the faces meet at edges and 8 corners.
2. **Hydrological catchments (Pfafstetter).** About 291,000 land catchments, drawn from terrain and river networks,
   independent of any model grid.
3. **Tiles.** A land tile is the **intersection of one grid cell with one catchment**. A grid cell crossed by 3
   catchments holds 3 land tiles, and a catchment spread over 5 cells is split into 5 tiles. Lakes, land ice
   and ocean get their own tiles. The Catchment land model runs **per land tile**: every model state,
   increment and ObsFcstAna `tilenum` refers to one of these 1,188,020 land tiles.

**Where the geometry comes from.** The BCS "raster" files describe each layer at 30 arc-seconds (about 1 km,
43200 × 21600 pixels). Each pixel holds the index of the grid cell, the catchment, or the tile it belongs to.
The tile file (`.nc4` / `.domain`) lists, per tile: type, area, centre of mass, grid cell `(i_indg, j_indg)`
and catchment `pfaf_index`. Reading a small window of a raster is cheap (memory-mapped), so we can draw the
true tile shapes for any region.

Run on Discover with the GEOSpyD Python 3.13 environment (`/usr/local/other/GEOSpyD/25.3.1-0/2025-10-07/envs/py3.13`).
Takes about a minute.

## 1. Imports and paths

In [ ]:
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

%matplotlib inline

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())

# BCS v14 geometry for CF0360 (from the experiment's exeinp: BCS_PATH + BCS_RESOLUTION)
BCS_GEOM = Path('/discover/nobackup/projects/gmao/bcs_shared/fvInput/ExtData/esm/tiles/v14/geometry/CF0360x6C_CF0360x6C')
TILE_FILE = BCS_GEOM / 'CF0360x6C_CF0360x6C-Pfafstetter.nc4'
RASTERS = {                                              # each pixel = index of ...
    'cell':  BCS_GEOM / 'rst' / 'CF0360x6C.rst',                              # ... the cubed-sphere grid cell
    'catch': BCS_GEOM / 'rst' / 'Pfafstetter.rst',                            # ... the Pfafstetter catchment
    'tile':  BCS_GEOM / 'rst' / 'CF0360x6C_CF0360x6C-Pfafstetter.rst',        # ... the tile
}

# An experiment on this tiling, used in section 5 to show where obs land
ROOT = Path('/gpfsm/dnb06/projects/p284/M21C_testing')
EXP = 'M21C_test_CF0360_allsp_superob025ml_OL6sc_xc03125'
OFA_DIR = ROOT / EXP / 'output' / 'CF0360x6C_GLOBAL' / 'ana' / 'ens_avg'

FIG_DIR = REPO_ROOT / 'projects' / 'M21C_land' / 'figures' / 'grid_and_tiles'
FIG_DIR.mkdir(parents=True, exist_ok=True)

N_FACE = 360                 # cells per face edge
R_EARTH_KM = 6371.0

## 2. Reading the tile table and the rasters

**Tile table.** One row per tile. (The experiment's `rc_out/*.domain` file is a copy of this table. The only
difference is that it marks some lake tiles as type 1019 instead of 19. Land tiles are identical.) Land tiles have `typ == 100` and come **first**, so land tile *k* (1-based) is
row *k*. That is why the model's tile dimension and ObsFcstAna's `tilenum` can index straight into this table.

**Grid-cell index.** `i_indg` runs 1–360 across a face and `j_indg` runs 1–2160 over the six faces stacked on top of
each other, so

- face = `(j_indg − 1) // 360 + 1` (1–6)
- row within the face = `(j_indg − 1) % 360 + 1`
- flat cell number, as stored in the cell raster = `(j_indg − 1) · 360 + i_indg`

The raster check below confirms that encoding.

**Raster files** are Fortran unformatted sequential: one record per latitude row, and each record is 43200 int32
values with a 4-byte length marker at both ends (so rows are 43202 values wide on disk). Row 0 is the
**southernmost** row (−90°) and column 0 starts at −180°, both 1/120° wide.

In [ ]:
with nc.Dataset(TILE_FILE) as d:
    T = pd.DataFrame({v: np.asarray(d[v][:]) for v in
                      ['typ', 'area', 'com_lon', 'com_lat', 'i_indg', 'j_indg', 'pfaf_index', 'frac_cell']})
    T.attrs = {a: d.getncattr(a) for a in d.ncattrs()}
T.index = np.arange(1, len(T) + 1)                        # 1-based tile number, as in the raster
T['face'] = (T.j_indg - 1) // N_FACE + 1
T['cell'] = (T.j_indg - 1) * N_FACE + T.i_indg
T['area_km2'] = T.area * R_EARTH_KM**2                    # area is on the unit sphere (radian^2)

LAND = T[T.typ == 100]
assert (LAND.index == np.arange(1, len(LAND) + 1)).all(), 'land tiles are expected to come first'
print(pd.Series({'all tiles': len(T), 'land tiles (typ 100)': len(LAND),
                 'lake (19)': (T.typ == 19).sum(), 'land ice (20)': (T.typ == 20).sum(),
                 'ocean (0)': (T.typ == 0).sum(), 'other (1019)': (T.typ == 1019).sum(),
                 'catchments': T.attrs.get('N_PfafCat')}).to_string())

NY, NX = 21600, 43200
RES = 1.0 / 120.0

def read_raster(name, lat0, lat1, lon0, lon1):
    """Window of a 30-arcsec raster: returns (values [rows S->N, cols W->E], lat_edges, lon_edges)."""
    mm = np.memmap(RASTERS[name], dtype='<i4', mode='r').reshape(NY, NX + 2)
    r0, r1 = int(np.floor((lat0 + 90) / RES)), int(np.ceil((lat1 + 90) / RES))
    c0, c1 = int(np.floor((lon0 + 180) / RES)), int(np.ceil((lon1 + 180) / RES))
    vals = np.array(mm[r0:r1, c0 + 1:c1 + 1])             # +1 skips the record-length marker
    return vals, -90 + RES * np.arange(r0, r1 + 1), -180 + RES * np.arange(c0, c1 + 1)

# Consistency checks on a small window, going through the tile table:
# 1. the cell raster must give the same grid cell as the tile table does for each pixel's tile.
# 2. the catchment raster uses a DIFFERENT catchment numbering than the tile table's pfaf_index
#    (they are not equal), but on land each raster id must correspond to exactly one pfaf_index.
#    That is all we need, because we only use the catchment raster to draw catchment boundaries.
t, _, _ = read_raster('tile', -21, -19, -58.5, -56.5)
c, _, _ = read_raster('cell', -21, -19, -58.5, -56.5)
p, _, _ = read_raster('catch', -21, -19, -58.5, -56.5)
land_px = T.typ.to_numpy()[t - 1] == 100
print('cell raster == tile table cell               :', np.mean(T.cell.to_numpy()[t - 1] == c))
pairs = pd.DataFrame({'raster': p[land_px], 'pfaf': T.pfaf_index.to_numpy()[t - 1][land_px]}).drop_duplicates()
print('catch raster ids with more than one pfaf_index:', int((pairs.groupby('raster').size() > 1).sum()),
      f'(of {pairs.raster.nunique()} catchments in the window)')

## 3. Some numbers: how many tiles per cell, and how big are they?

- **Land tiles per grid cell**: a land cell holds 1 tile if a single catchment covers it, more where catchment
  boundaries cross it.
- **Land tile area**: a full cell is about 28 × 28 km ≈ 800 km², but most tiles are only part of a cell.

In [ ]:
tiles_per_cell = LAND.groupby('cell').size()
fig, axs = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
axs[0].hist(tiles_per_cell, bins=np.arange(0.5, 15.5, 1), color='0.4')
axs[0].set_xlabel('land tiles per grid cell'); axs[0].set_ylabel('number of grid cells')
axs[0].set_title(f'{tiles_per_cell.size:,} cells with land; median {tiles_per_cell.median():.0f}, '
                 f'mean {tiles_per_cell.mean():.2f} tiles/cell')
axs[1].hist(LAND.area_km2, bins=np.logspace(-1, 3.2, 60), color='0.4')
axs[1].set_xscale('log'); axs[1].set_xlabel('land tile area [km²]'); axs[1].set_ylabel('number of tiles')
axs[1].set_title(f'median {LAND.area_km2.median():.0f} km², mean {LAND.area_km2.mean():.0f} km²; '
                 f'{(LAND.area_km2 < 50).mean():.0%} of tiles < 50 km²')
fig.savefig(FIG_DIR / 'tiles_per_cell_and_area.png', dpi=150)
plt.show()

## 4. The six faces of the cube, globally

We read every 20th pixel of the cell raster (about 10 km → 0.17° spacing), which is enough to see the faces.

- **Left:** each colour is one face.
- **Right:** a checkerboard of 30 × 30-cell blocks (so each face is 12 × 12 blocks). The blocks show how the grid
  lines bend and how the cells stay about the same size, instead of converging at the poles like lat/lon cells.

Each face is labelled at its centre cell. Look for the **corners** where three faces meet: cells are most
distorted there, and a corner can fall on land.

In [ ]:
STRIDE = 20
mm = np.memmap(RASTERS['cell'], dtype='<i4', mode='r').reshape(NY, NX + 2)
cellg = np.array(mm[::STRIDE, 1:-1:STRIDE])                   # S->N, W->E
lat_e = -90 + RES * STRIDE * np.arange(cellg.shape[0] + 1)
lon_e = -180 + RES * STRIDE * np.arange(cellg.shape[1] + 1)
face = (cellg - 1) // (N_FACE * N_FACE) + 1
ii = (cellg - 1) % N_FACE                                     # 0..359 within the face
jj = ((cellg - 1) // N_FACE) % N_FACE                         # 0..359 within the face
checker = ((ii // 30 + jj // 30) % 2) + 2 * (face - 1)        # two shades per face

face_cmap = mcolors.ListedColormap(plt.get_cmap('tab10').colors[:6])
pair = []
for c in plt.get_cmap('tab10').colors[:6]:
    pair += [tuple(0.55 + 0.45 * np.array(c)), c]            # light / full shade of each face colour
check_cmap = mcolors.ListedColormap(pair)

fig, axs = plt.subplots(1, 2, figsize=(18, 5.5), subplot_kw={'projection': ccrs.Robinson()},
                        constrained_layout=True)
axs[0].pcolormesh(lon_e, lat_e, face, cmap=face_cmap, vmin=0.5, vmax=6.5, transform=ccrs.PlateCarree(),
                  rasterized=True)
axs[1].pcolormesh(lon_e, lat_e, checker, cmap=check_cmap, vmin=-0.5, vmax=11.5, transform=ccrs.PlateCarree(),
                  rasterized=True)
for ax in axs:
    ax.set_global()
    ax.coastlines(linewidth=0.5, color='k')
# label each face at the location of its centre cell (i = j = 180 within the face)
for f in range(1, 7):
    ctr = T[(T.face == f) & (T.i_indg == 180) & (((T.j_indg - 1) % N_FACE) == 179)]
    if len(ctr):
        axs[0].text(ctr.com_lon.iloc[0], ctr.com_lat.iloc[0], f'face {f}', transform=ccrs.PlateCarree(),
                    ha='center', va='center', fontsize=12, weight='bold',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
axs[0].set_title('Cubed-sphere faces (CF0360: 6 × 360 × 360 cells)')
axs[1].set_title('Blocks of 30 × 30 cells (12 × 12 blocks per face)')
fig.savefig(FIG_DIR / 'cube_faces_global.png', dpi=150)
plt.show()

## 5. Zoom: grid cells, catchments and tiles in one small region

For a 2° × 2° window, three panels show the three layers at the raster's 1 km resolution, each id in a random
colour so neighbours stand apart:

1. **Grid cells**: the regular-looking CF0360 cells (≈ 28 km).
2. **Catchments**: irregular, following terrain and rivers. Some are much larger than a cell, some much smaller.
3. **Tiles** = cell ∩ catchment, with **black lines = grid-cell boundaries**, thin white lines = catchment
   boundaries, and dots = tile centres of mass. Each piece between the lines is one tile. Non-land tiles
   (lakes, ice, ocean) are drawn white.

`plot_layers` works for any window, so change `ZOOMS` to look elsewhere.

In [ ]:
def random_colors(ids, seed=0):
    """Map arbitrary integer ids to shuffled colour indices, so adjacent ids get different colours."""
    uniq, inv = np.unique(ids, return_inverse=True)
    perm = np.random.default_rng(seed).permutation(uniq.size)
    return (perm[inv] % 20).reshape(ids.shape)


def boundaries(a):
    """True on pixels whose id differs from the pixel to the east or north (i.e. on a boundary)."""
    e = np.zeros(a.shape, bool)
    e[:, 1:] |= a[:, 1:] != a[:, :-1]
    e[1:, :] |= a[1:, :] != a[:-1, :]
    return e


def draw_mask(ax, mask, lat_e, lon_e, color):
    """Overlay the True pixels of a boolean mask in one colour."""
    ax.pcolormesh(lon_e, lat_e, np.ma.masked_where(~mask, mask.astype(float)),
                  cmap=mcolors.ListedColormap([color]), vmin=0, vmax=1, rasterized=True)


def plot_layers(name, lat0, lat1, lon0, lon1):
    cell, lat_e, lon_e = read_raster('cell', lat0, lat1, lon0, lon1)
    catch, _, _ = read_raster('catch', lat0, lat1, lon0, lon1)
    tile, _, _ = read_raster('tile', lat0, lat1, lon0, lon1)
    is_land = T.typ.to_numpy()[tile - 1] == 100

    fig, axs = plt.subplots(1, 3, figsize=(19, 6.6), constrained_layout=True)
    tab20 = plt.get_cmap('tab20')
    axs[0].pcolormesh(lon_e, lat_e, random_colors(cell), cmap=tab20, vmin=0, vmax=19, rasterized=True)
    axs[1].pcolormesh(lon_e, lat_e, random_colors(catch, 1), cmap=tab20, vmin=0, vmax=19, rasterized=True)
    axs[2].pcolormesh(lon_e, lat_e, np.ma.masked_where(~is_land, random_colors(tile, 2)), cmap=tab20,
                      vmin=0, vmax=19, rasterized=True)
    draw_mask(axs[2], boundaries(catch), lat_e, lon_e, 'white')
    draw_mask(axs[2], boundaries(cell), lat_e, lon_e, 'black')
    inside = LAND[(LAND.com_lat.between(lat0, lat1)) & (LAND.com_lon.between(lon0, lon1))]
    axs[2].plot(inside.com_lon, inside.com_lat, 'k.', ms=2.5)

    n_cell = np.unique(cell).size
    n_catch = np.unique(catch).size
    n_tile = np.unique(tile[is_land]).size
    axs[0].set_title(f'(a) grid cells: {n_cell}')
    axs[1].set_title(f'(b) catchments: {n_catch}')
    axs[2].set_title(f'(c) land tiles: {n_tile}  (black = cells, white = catchments)')
    for ax in axs:
        ax.set_aspect(1 / np.cos(np.radians(0.5 * (lat0 + lat1))))
        ax.set_xlabel('lon'); ax.set_ylabel('lat')
    fig.suptitle(f'{name}: {lat0}..{lat1}N, {lon0}..{lon1}E', fontsize=14)
    fig.savefig(FIG_DIR / f'layers_{name.replace(" ", "_")}.png', dpi=150)
    plt.show()


ZOOMS = {
    'Pantanal':   (-21.0, -19.0, -58.5, -56.5),    # wetland floodplain: large flat catchments
    'Kansas':     (37.0, 39.0, -99.0, -97.0),       # plains: catchments roughly cell-sized
    'Great Slave Lake': (60.5, 62.5, -116.5, -112.5),  # lake tiles, high latitude
}
for name, box in ZOOMS.items():
    plot_layers(name, *box)

## 6. How observations land on tiles

**The assignment rule** (from `GEOSldas_GridComp`: `get_pert_grid` in `LDAS_PertRoutines.F90`,
`get_tile_num_from_latlon` in `LDAS_TileCoordRoutines.F90`, called from `clsm_ensupd_enkf_update.F90`).
Read from the develop-based source checkout, and verified against the run output in section 6b.

1. For cube-sphere tile spaces, obs are not located on the cube grid. GEOSldas uses a regular lat/lon
   **"pert grid"** instead, hard-wired to 4N × 3N cells for a CN cube: `DE1440xPE1080` for C360, i.e.
   **0.25° in longitude × 1/6° in latitude** (red dotted lines below). The same grid carries the ensemble perturbations.
2. Each land tile is listed under the pert-grid cell that contains its **centre of mass** (not its whole area).
3. An obs goes to the tile, among those listed in the obs's pert-grid cell, whose centre of mass is **nearest**
   (distance |Δlon| + |Δlat| in degrees). So the obs is **not necessarily inside its tile**: it can go to a
   neighbouring tile whose centre is nearer. (The obs is dropped only if it is outside the tile's bounding box
   *and* farther than a sensor-specific max distance.)

The obs grids are yet another layer:

- SMAP / SMOS Tb and CYGNSS: **EASE2 36 km** cells (black dashed lines). Each obs sits at a cell centre.
- H SAF super-obs: **0.25° lat/lon** cells. Each obs is at the mean location of the raw obs averaged into it.

A 36 km obs covers several CF0360 tiles but is a single point, so it goes to **one** tile. The panels colour the
tiles that received at least one obs in 3 days of the allsp run. Most tiles never receive a SMAP obs, which is
why per-tile maps of obs statistics look speckled, and why `cf0360_allsp_may2019_ofa_maps.ipynb` maps the
statistics on each sensor's own obs grid.

(The EnKF spreads the information to neighbouring tiles through the ensemble covariances and the
localization. Tiles without an obs still get increments. They just have no O-F of their own.)

In [ ]:
SPECIES_GROUP = {5: 'SMAP Tb', 6: 'SMAP Tb', 7: 'SMAP Tb', 8: 'SMAP Tb', 12: 'HSAF SM', 13: 'HSAF SM', 14: 'HSAF SM'}
DAYS = ['20190515', '20190516', '20190517']

EASE_PATH = REPO_ROOT / 'common' / 'python' / 'plotting' / 'ease_grids'
m36_lat = np.fromfile(EASE_PATH / 'EASE2_M36km.lats.964x406x1.double').reshape(406, 964)[:, 0][::-1]
m36_lat_edges = np.concatenate([[-90.0], 0.5 * (m36_lat[:-1] + m36_lat[1:]), [90.0]])
m36_lon_edges = np.linspace(-180.0, 180.0, 965)

# The lat/lon "pert grid" used to assign obs to tiles: 4N x 3N cells for a CN cube (DE = dateline edge,
# PE = pole edge, i.e. cell EDGES start at -180 and -90)
PERT_NLON, PERT_NLAT = 4 * N_FACE, 3 * N_FACE
PERT_LON_EDGES = np.linspace(-180.0, 180.0, PERT_NLON + 1)
PERT_LAT_EDGES = np.linspace(-90.0, 90.0, PERT_NLAT + 1)

frames = []
for day in DAYS:
    for f in sorted(glob.glob(str(OFA_DIR / f'Y{day[:4]}' / f'M{day[4:6]}' / f'*ObsFcstAna.{day}_*.nc4'))):
        with nc.Dataset(f) as d:
            frames.append(pd.DataFrame({'species': d['species'][:], 'tilenum': d['tilenum'][:],
                                        'lat': d['lat'][:], 'lon': d['lon'][:]}))
OBS = pd.concat(frames)
OBS['group'] = OBS.species.map(SPECIES_GROUP)
OBS = OBS.dropna(subset=['group'])


def plot_obs_on_tiles(name, lat0, lat1, lon0, lon1):
    tile, lat_e, lon_e = read_raster('tile', lat0, lat1, lon0, lon1)
    cell, _, _ = read_raster('cell', lat0, lat1, lon0, lon1)
    is_land = T.typ.to_numpy()[tile - 1] == 100
    box = OBS[OBS.lat.between(lat0, lat1) & OBS.lon.between(lon0, lon1)]

    fig, axs = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
    for ax, g, color in zip(axs, ['SMAP Tb', 'HSAF SM'], ['tab:blue', 'tab:orange']):
        obs = box[box.group == g]
        hit = np.isin(tile, obs.tilenum.unique()) & is_land
        shade = np.where(~is_land, 0, np.where(hit, 2, 1))          # 0 = not land, 1 = no obs, 2 = obs
        ax.pcolormesh(lon_e, lat_e, shade, cmap=mcolors.ListedColormap(['white', '0.85', color]),
                      vmin=-0.5, vmax=2.5, rasterized=True)
        draw_mask(ax, boundaries(np.where(is_land, tile, -1)), lat_e, lon_e, '0.55')   # tile outlines
        draw_mask(ax, boundaries(cell), lat_e, lon_e, 'black')                         # grid cells
        if g == 'SMAP Tb':                                 # obs grid: EASE2 M36
            for x in m36_lon_edges[(m36_lon_edges > lon0) & (m36_lon_edges < lon1)]:
                ax.axvline(x, color='k', ls='--', lw=1)
            for y in m36_lat_edges[(m36_lat_edges > lat0) & (m36_lat_edges < lat1)]:
                ax.axhline(y, color='k', ls='--', lw=1)
        for x in PERT_LON_EDGES[(PERT_LON_EDGES > lon0) & (PERT_LON_EDGES < lon1)]:     # pert grid
            ax.axvline(x, color='red', ls=':', lw=1.2)
        for y in PERT_LAT_EDGES[(PERT_LAT_EDGES > lat0) & (PERT_LAT_EDGES < lat1)]:
            ax.axhline(y, color='red', ls=':', lw=1.2)
        ax.plot(obs.lon, obs.lat, 'k+', ms=7, mew=1.2)
        n_land = np.unique(tile[is_land]).size
        n_hit = np.unique(tile[hit]).size
        ax.set_title(f'{g}: {len(obs)} obs in {len(DAYS)} days hit {n_hit} of {n_land} land tiles')
        ax.set_xlim(lon0, lon1); ax.set_ylim(lat0, lat1)
        ax.set_aspect(1 / np.cos(np.radians(0.5 * (lat0 + lat1))))
    fig.suptitle(f'{name}: tiles that received obs (coloured), CF0360 cells (black), '
                 f'pert grid (red dotted), EASE2 M36 (dashed), obs (+)', fontsize=13)
    fig.savefig(FIG_DIR / f'obs_on_tiles_{name.replace(" ", "_")}.png', dpi=150)
    plt.show()


plot_obs_on_tiles('Kansas', 37.5, 38.5, -98.5, -97.0)

### 6b. Check: reproduce the assignment from the rule

For a sample of obs (all species, one day), apply the rule from section 6 in Python and compare with the
`tilenum` GEOSldas wrote to ObsFcstAna. The max-distance / bounding-box check is not reproduced (it only
drops obs, it never changes which tile is chosen).

When this was written, the rule reproduced about 99% overall, and 99–100% for SMAP, CYGNSS, H SAF and MODIS.
**SMOS matched only about 85%**, not yet investigated. A plausible cause: the reader assigns the tile from the
original SMOS location, while ObsFcstAna stores a slightly different (grid) location.

In [ ]:
land_lat, land_lon = LAND.com_lat.to_numpy(), LAND.com_lon.to_numpy()
tile_key = (np.floor((land_lat + 90) / (180 / PERT_NLAT)).astype(int) * PERT_NLON
            + np.floor((land_lon + 180) / (360 / PERT_NLON)).astype(int))
order = np.argsort(tile_key, kind='stable')
sorted_key = tile_key[order]

frames = []
for f in sorted(glob.glob(str(OFA_DIR / 'Y2019' / 'M05' / '*ObsFcstAna.20190515_*.nc4'))):
    with nc.Dataset(f) as d:
        frames.append(pd.DataFrame({'species': d['species'][:], 'tilenum': d['tilenum'][:],
                                    'lat': d['lat'][:], 'lon': d['lon'][:]}))
sample = pd.concat(frames).sample(20000, random_state=0)

obs_key = (np.floor((sample.lat.to_numpy() + 90) / (180 / PERT_NLAT)).astype(int) * PERT_NLON
           + np.floor((sample.lon.to_numpy() + 180) / (360 / PERT_NLON)).astype(int))
pred = np.full(len(sample), -1)
for n, (k, la, lo) in enumerate(zip(obs_key, sample.lat.to_numpy(), sample.lon.to_numpy())):
    cand = order[np.searchsorted(sorted_key, k):np.searchsorted(sorted_key, k, side='right')]
    if cand.size:
        pred[n] = cand[np.argmin(np.abs(land_lon[cand] - lo) + np.abs(land_lat[cand] - la))] + 1
sample['rule_matches'] = pred == sample.tilenum.to_numpy()
print(f'rule reproduces tilenum for {sample.rule_matches.mean():.1%} of {len(sample):,} sampled obs')
print(sample.groupby('species').rule_matches.mean().round(3).to_string())

## 7. From tiles to cube cells: reproducing the first step of the gridded output

The gridded `_Nx` output files (e.g. `tavg24_2d_lnd_Nx`) are made by MAPL History in **two steps**
(`HISTORY.rc`: `regrid_exch`, `regrid_name`, `grid_label`):

1. **Tiles → cube cells** (`MAPL_LocStreamTransformT2G` in `MAPL_LocStreamMod.F90`). Each cube cell gets the
   weighted mean of the land tiles in it, where each tile's weight is its `frac_cell` from the tile file
   (tile area / cell area). Tiles with an undefined value are skipped, and cells with no land tile become undefined.
2. **Cube cells → 0.5° lat/lon** (`PC720x361-DC`) by **bilinear** interpolation, the MAPL History default,
   because `regrid_method` is commented out in `HISTORY.rc`.

Step 2 smooths further and mixes values across coasts, and it is all the OL6 run wrote. With the tile table we
can do **step 1 ourselves** for any tile-space output (`catch_progn_incr`, `inst3_1d_lndfcstana_Nt`,
restarts) and draw the result on the **true cell shapes** from the cell raster, with no interpolation.

`tiles_to_cells` below is the same formula as MAPL's T2G: `Σ w·x / Σ w` over the land tiles in each cell,
with `w = frac_cell`.

In [ ]:
N_CELL = 6 * N_FACE * N_FACE

# frac_cell should add up to ~1 over ALL tiles (land, lake, ice, ocean) of a cell: a quick sanity check
frac_sum = np.bincount(T.cell.to_numpy() - 1, weights=T.frac_cell.to_numpy(), minlength=N_CELL)
print(f'sum of frac_cell per cell over all tile types: median {np.median(frac_sum):.4f}, '
      f'1st-99th pct {np.percentile(frac_sum, 1):.4f}-{np.percentile(frac_sum, 99):.4f}')

LAND_CELL = LAND.cell.to_numpy() - 1          # 0-based cell index of each land tile
LAND_W = LAND.frac_cell.to_numpy()


def tiles_to_cells(values):
    # Area-weighted mean of land-tile values per cube cell (same as MAPL T2G); NaN where no valid land tile.
    ok = np.isfinite(values)
    num = np.bincount(LAND_CELL[ok], weights=LAND_W[ok] * values[ok], minlength=N_CELL)
    den = np.bincount(LAND_CELL[ok], weights=LAND_W[ok], minlength=N_CELL)
    out = np.full(N_CELL, np.nan)
    out[den > 0] = num[den > 0] / den[den > 0]
    return out


def read_tile_var(collection, stamp, var):
    # One variable of a tile-space (_Nt / catch_progn_incr) output file of EXP; fill values -> NaN.
    f = (ROOT / EXP / 'output' / 'CF0360x6C_GLOBAL' / 'cat' / 'ens_avg' / f'Y{stamp[:4]}' / f'M{stamp[4:6]}'
         / f'{EXP}.{collection}.{stamp}.nc4')
    with nc.Dataset(f) as d:
        return np.ma.filled(d[var][0, :].astype(np.float64), np.nan)


# Example field: analysis root-zone soil moisture at 2019-05-31 00z (instantaneous, tile space)
rz_tile = read_tile_var('inst3_1d_lndfcstana_Nt', '20190531_0000z', 'RZMC_ANA')
rz_cell = tiles_to_cells(rz_tile)
print(f'{np.isfinite(rz_tile).sum():,} land tiles -> {np.isfinite(rz_cell).sum():,} cube cells with a value')

### 7a. Global map on cube cells

Each pixel of the (every-20th-pixel) cell raster from section 4 simply takes its cell's value, so this is the
cube-cell field on the true cell shapes, drawn in Robinson.

In [ ]:
def cells_to_raster(cell_values, cell_raster):
    # Paint per-cell values onto a cell-index raster (raster values are 1-based cell numbers).
    return cell_values[cell_raster - 1]

fig = plt.figure(figsize=(14, 6.5), constrained_layout=True)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
mesh = ax.pcolormesh(lon_e, lat_e, cells_to_raster(rz_cell, cellg), cmap='YlGnBu', vmin=0, vmax=0.8,
                     transform=ccrs.PlateCarree(), rasterized=True, zorder=1)
ax.coastlines(linewidth=0.3, zorder=2)
fig.colorbar(mesh, ax=ax, orientation='horizontal', shrink=0.5, pad=0.02, label='RZMC_ANA [m³/m³]')
ax.set_title(f'{EXP}: RZMC_ANA 2019-05-31 00z, land tiles averaged to CF0360 cube cells (frac_cell weights)')
fig.savefig(FIG_DIR / 'rzmc_cube_cells_global.png', dpi=150)
plt.show()

### 7b. Zoom: tiles vs cube cells vs the 0.5° output file

The same field at three stages, over the Pantanal where DA saturated a few wetland tiles in May (see the
all-species write-up):

1. **tiles**: `RZMC_ANA` per land tile (true tile shapes from the tile raster);
2. **cube cells**: after step 1 (`tiles_to_cells`), on the true cell shapes;
3. **0.5° output file**: `RZMC` from `tavg24_2d_lnd_Nx` for 2019-05-30 (the daily mean ending at 05-31 00z), after
   step 2 as well. This is a daily mean, not the 00z value, so it is not an exact match with panels 1–2. The point is how much
   spatial detail each step removes.

In [ ]:
def plot_three_stages(name, lat0, lat1, lon0, lon1, vmin=0.3, vmax=0.95):
    tile, lat_z, lon_z = read_raster('tile', lat0, lat1, lon0, lon1)
    cell, _, _ = read_raster('cell', lat0, lat1, lon0, lon1)
    is_land = T.typ.to_numpy()[tile - 1] == 100
    tile_field = np.where(is_land, rz_tile[np.clip(tile - 1, 0, len(rz_tile) - 1)], np.nan)
    cell_field = np.where(is_land, rz_cell[cell - 1], np.nan)

    f = (ROOT / EXP / 'output' / 'CF0360x6C_GLOBAL' / 'cat' / 'ens_avg' / 'Y2019' / 'M05'
         / f'{EXP}.tavg24_2d_lnd_Nx.20190530_1200z.nc4')
    with nc.Dataset(f) as d:
        glat, glon = np.asarray(d['lat'][:]), np.asarray(d['lon'][:])
        g = np.ma.filled(d['RZMC'][0].astype(float), np.nan)
    # 0.5 deg cell edges around the file's cell centres
    glat_e = np.concatenate([[glat[0] - 0.25], 0.5 * (glat[:-1] + glat[1:]), [glat[-1] + 0.25]])
    glon_e = np.concatenate([[glon[0] - 0.25], 0.5 * (glon[:-1] + glon[1:]), [glon[-1] + 0.25]])

    fig, axs = plt.subplots(1, 3, figsize=(19, 6.6), constrained_layout=True)
    kw = dict(cmap='YlGnBu', vmin=vmin, vmax=vmax, rasterized=True)
    axs[0].pcolormesh(lon_z, lat_z, tile_field, **kw)
    axs[1].pcolormesh(lon_z, lat_z, cell_field, **kw)
    mesh = axs[2].pcolormesh(glon_e, glat_e, g, **kw)
    for ax in axs[:2]:
        draw_mask(ax, boundaries(cell), lat_z, lon_z, 'black')          # cube-cell outlines
    axs[0].set_title('(a) land tiles, RZMC_ANA 05-31 00z')
    axs[1].set_title('(b) cube cells (frac_cell-weighted tile mean)')
    axs[2].set_title('(c) tavg24_2d_lnd_Nx, 0.5°, daily mean 05-30')
    for ax in axs:
        ax.set_xlim(lon0, lon1); ax.set_ylim(lat0, lat1)
        ax.set_aspect(1 / np.cos(np.radians(0.5 * (lat0 + lat1))))
    fig.colorbar(mesh, ax=axs.tolist(), orientation='horizontal', shrink=0.4,
                 label='root-zone soil moisture [m³/m³]')
    fig.suptitle(f'{name}: tile → cube cell → 0.5° lat/lon', fontsize=14)
    fig.savefig(FIG_DIR / f'three_stages_{name.replace(" ", "_")}.png', dpi=150)
    plt.show()


plot_three_stages('Pantanal', -21.0, -19.0, -58.5, -56.5)

## 8. Where the model output lives in all this

| Output | Space | Notes |
|---|---|---|
| restarts, `catch_progn_incr`, `inst3_1d_lndfcstana_Nt` | land tiles (1,188,020) | the `tile` dimension is the land rows of the tile table |
| `ObsFcstAna` | obs, each with a `tilenum` | 1-based land tile number; `lat`/`lon` = obs location, not the tile centre |
| `tavg24_2d_lnd_Nx` and other `_Nx` | 0.5° lat/lon grid | tiles → cube cells (`frac_cell`-weighted mean), then bilinear to 0.5° (section 7) |
| `*.til`, `*.rst` (BCS) | tile table and 30″ rasters | the geometry used in this notebook |